# Explicabilidade BGE — resultados globais e auditoria por conversa

Notebook dedicado **somente** à explicabilidade hierárquica do Transformer com embeddings BGE.

Ele não executa BGE nem o classificador: lê os artefatos consolidados produzidos pelo estudo e reproduz todas as análises já validadas:

- cobertura e denominadores `N_k`;
- comprehensiveness e sufficiency distance, com média, dispersão e IC95%;
- saturação por número e por proporção de mensagens;
- `k95` e massa acumulada de evidência;
- distribuições globais e por TP, TN, FP e FN;
- n-grams selecionados versus controles aleatórios;
- estabilidade entre seeds 42, 52 e 62;
- auditoria completa de conversas de validação;
- explorador de mensagens, trechos, fidelidade, controles e estabilidade.

As métricas globais usam `decision_supporting`; o explorador também mostra `absolute_influence`. Todos os valores globais são mantidos separados dos valores de uma conversa específica.


## Leitura rápida das métricas

- **Comprehensiveness:** quanto o suporte à classe prevista cai ao remover as evidências escolhidas. Maior é melhor para uma explicação causal; valores assinados negativos indicam interação ou evidência contrária.
- **Sufficiency distance:** distância entre a saída original e a saída usando somente as evidências escolhidas. Menor é melhor e a versão absoluta nunca é negativa.
- **IC95%:** intervalo de confiança bootstrap para a média entre conversas; não descreve a variação individual, que aparece nos boxplots.
- **`N_k`:** conversas com pelo menos `k` mensagens. **`metric_N_k`:** conversas com pelo menos `k` evidências válidas para a política avaliada.
- **`k95`:** primeiro `k` que alcança 95% do maior comprehensiveness observado naquela conversa.
- **Massa de evidência:** soma dos impactos individuais. Ela é monotônica dentro de cada conversa, mas não substitui a reavaliação contrafactual conjunta.

Nas curvas por fração da conversa, cada ponto agrega uma faixa percentual distinta e pode ter outro conjunto de conversas. Por isso a curva proporcional não precisa ter o mesmo formato da curva por `k` absoluto.


In [ ]:
from pathlib import Path
import json
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 180)

def find_repository_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'backend').is_dir() and (candidate / 'backend' / 'notebooks').is_dir():
            return candidate
    # Útil quando o kernel é iniciado em outro diretório, mas o caminho padrão existe.
    fallback = Path(r'C:/Users/Master/Desktop/Recod_ia/HorusEmbbedingPlusML')
    if fallback.exists():
        return fallback
    raise FileNotFoundError('Não foi possível localizar a raiz do repositório HorusEmbbedingPlusML.')

REPO_ROOT = find_repository_root()
default_root = REPO_ROOT / 'backend' / 'experiment_results' / 'transformer' / 'studies' / 'hierarchical_explainability'
HIER_STUDY_ROOT = Path(os.environ.get('HORUS_BGE_EXPLAINABILITY_ROOT', default_root)).resolve()

def read_hierarchical_parquet(relative, required=False):
    path = HIER_STUDY_ROOT / relative
    if not path.exists():
        message = f'Artefato ausente: {path}'
        if required:
            raise FileNotFoundError(message)
        print(message)
        return pd.DataFrame()
    return pd.read_parquet(path)

def concat_available(frames):
    available = [frame for frame in frames if not frame.empty]
    return pd.concat(available, ignore_index=True) if available else pd.DataFrame()

manifest_path = HIER_STUDY_ROOT / 'manifest.json'
HIER_MANIFEST = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}

hierarchical_coverage = read_hierarchical_parquet('aggregates/coverage.parquet')
hierarchical_curves = read_hierarchical_parquet('aggregates/fidelity_curves.parquet')
hierarchical_metrics = read_hierarchical_parquet('aggregates/metric_distributions.parquet')
hierarchical_saturation_curves = read_hierarchical_parquet('aggregates/saturation_curves.parquet')
hierarchical_saturation_fraction_curves = read_hierarchical_parquet('aggregates/saturation_fraction_curves.parquet')
hierarchical_saturation_points = read_hierarchical_parquet('aggregates/saturation_points.parquet')
hierarchical_saturation_conversations = read_hierarchical_parquet('aggregates/saturation_conversations.parquet')
hierarchical_validation_saturation = read_hierarchical_parquet('validation/fidelity_saturation.parquet')
hierarchical_control_summary = read_hierarchical_parquet('aggregates/random_control_summary.parquet')
hierarchical_controls = concat_available([
    read_hierarchical_parquet(f'{split}/random_controls.parquet')
    for split in ('test_internal', 'validation')
])
hierarchical_stability = concat_available([
    read_hierarchical_parquet(f'{split}/stability.parquet')
    for split in ('test_internal', 'validation')
])
hierarchical_messages = concat_available([
    read_hierarchical_parquet(f'{split}/message_effects.parquet')
    for split in ('test_internal', 'validation')
])
hierarchical_spans = concat_available([
    read_hierarchical_parquet(f'{split}/span_effects.parquet')
    for split in ('test_internal', 'validation')
])

HIER_SPLIT_COLORS = {'test_internal': '#2563eb', 'validation': '#f97316'}
HIER_CATEGORY_COLORS = {'TP': '#16a34a', 'TN': '#2563eb', 'FP': '#dc2626', 'FN': '#9333ea'}

print(f'Repositório: {REPO_ROOT}')
print(f'Artefatos:   {HIER_STUDY_ROOT}')
print(f'Manifesto:   {"carregado" if HIER_MANIFEST else "ausente"}')


## Inventário e proveniência dos artefatos

In [ ]:
artifact_inventory = pd.DataFrame([
    {'artefato': 'coverage', 'linhas': len(hierarchical_coverage), 'colunas': len(hierarchical_coverage.columns)},
    {'artefato': 'fidelity_curves', 'linhas': len(hierarchical_curves), 'colunas': len(hierarchical_curves.columns)},
    {'artefato': 'metric_distributions', 'linhas': len(hierarchical_metrics), 'colunas': len(hierarchical_metrics.columns)},
    {'artefato': 'saturation_curves', 'linhas': len(hierarchical_saturation_curves), 'colunas': len(hierarchical_saturation_curves.columns)},
    {'artefato': 'saturation_fraction_curves', 'linhas': len(hierarchical_saturation_fraction_curves), 'colunas': len(hierarchical_saturation_fraction_curves.columns)},
    {'artefato': 'saturation_points', 'linhas': len(hierarchical_saturation_points), 'colunas': len(hierarchical_saturation_points.columns)},
    {'artefato': 'message_effects', 'linhas': len(hierarchical_messages), 'colunas': len(hierarchical_messages.columns)},
    {'artefato': 'span_effects', 'linhas': len(hierarchical_spans), 'colunas': len(hierarchical_spans.columns)},
    {'artefato': 'random_controls', 'linhas': len(hierarchical_controls), 'colunas': len(hierarchical_controls.columns)},
    {'artefato': 'stability', 'linhas': len(hierarchical_stability), 'colunas': len(hierarchical_stability.columns)},
])
display(artifact_inventory)
if HIER_MANIFEST:
    display(pd.json_normalize(HIER_MANIFEST, sep='.').T.rename(columns={0: 'valor'}))


### 1. Cobertura global e denominadores por Top-k

In [ ]:
if not hierarchical_coverage.empty:
    display(hierarchical_coverage.sort_values(['split', 'top_k']).style.format({'N_k': '{:,.0f}', 'metric_N_k': '{:,.0f}'}))
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
    for ax, split in zip(axes, ('test_internal', 'validation')):
        view = hierarchical_coverage[hierarchical_coverage.split.eq(split)]
        ax.bar(view.top_k - .16, view.N_k, width=.32, color=HIER_SPLIT_COLORS[split], label='N_k: conversas com ≥ k turnos')
        ax.bar(view.top_k + .16, view.metric_N_k, width=.32, color='#64748b', label='N métrico: ≥ k evidências válidas')
        ax.set(title=f'Cobertura global — {split} — seed 42', xlabel='Top-k', ylabel='Conversas')
        ax.legend(); ax.grid(axis='y', alpha=.2)
    plt.tight_layout(); plt.show()


### 2. Comprehensiveness global — média e IC95% bootstrap

In [ ]:
def plot_hierarchical_curve(metric, title, ylabel):
    if hierarchical_curves.empty:
        return
    view = hierarchical_curves[(hierarchical_curves.metric.eq(metric)) & (hierarchical_curves.aggregation.eq('population'))]
    if view.empty:
        return
    plt.figure(figsize=(9, 5))
    for split, group in view.groupby('split'):
        group = group.sort_values('top_k')
        plt.plot(group.top_k, group['mean'], marker='o', color=HIER_SPLIT_COLORS[split], label=f'{split} (N por ponto em tabela)')
        plt.fill_between(group.top_k, group.ci95_low, group.ci95_high, color=HIER_SPLIT_COLORS[split], alpha=.18)
    plt.axhline(0, color='black', lw=1)
    plt.title(f'{title}\nGLOBAL | decision_supporting | seed 42')
    plt.xlabel('Quantidade acumulada de mensagens'); plt.ylabel(ylabel)
    plt.legend(title='Split / IC95%'); plt.grid(alpha=.25); plt.show()
    display(view[['split', 'top_k', 'n', 'mean', 'std', 'sem', 'ci95_low', 'ci95_high', 'negative_rate']].sort_values(['split', 'top_k']))

plot_hierarchical_curve('comprehensiveness', 'Comprehensiveness por Top-k', 'Queda na probabilidade da classe prevista')


### 3. Sufficiency distance global — média e IC95% bootstrap

In [ ]:
plot_hierarchical_curve(
    'sufficiency_distance',
    'Sufficiency distance por Top-k',
    '|P(classe prevista | conversa) − P(classe prevista | evidências)|',
)


### 3A. Saturação da explicação além das Top-6

O **logit** é o valor bruto produzido pelo classificador antes da sigmoid. Em probabilidades próximas de 0 ou 1, uma mudança relevante no logit pode parecer quase nula após a sigmoid. Por isso, a curva abaixo usa `comprehensiveness_logit_aligned`: positivo sempre significa queda do suporte à classe prevista após remover mensagens.

`k95` é o primeiro número de mensagens que alcança 95% do maior comprehensiveness em logit observado para aquela conversa. Ele também é apresentado como fração do total de mensagens. A sufficiency distance mede a distância entre a decisão original e a decisão usando somente as evidências; **menor é melhor**.

As curvas abaixo comparam `test_internal` e `validation`. Atenção: em cada `k`, entram somente conversas com pelo menos `k` evidências disponíveis. Portanto, `N` diminui com `k`; uma queda brusca da média pode combinar interação real entre mensagens com **mudança de composição da amostra**. O painel de cobertura deve ser lido junto com as métricas.


In [ ]:
if not hierarchical_saturation_curves.empty:
    decision = hierarchical_saturation_curves[
        hierarchical_saturation_curves.ranking_policy.eq('decision_supporting')
    ]
    fig, axes = plt.subplots(1, 3, figsize=(21, 5))
    for ax, metric, title, ylabel in [
        (axes[0], 'comprehensiveness_logit_aligned', 'Comprehensiveness conjunto em logit', 'Queda alinhada do logit'),
        (axes[1], 'sufficiency_distance_logit', 'Sufficiency distance em logit', '|logit original − logit somente evidências|'),
    ]:
        for split, view in decision[decision.metric.eq(metric)].groupby('split'):
            view = view.sort_values('top_k')
            ax.plot(view.top_k, view['mean'], color=HIER_SPLIT_COLORS[split], marker='o', ms=3, label=f'{split} — média')
            ax.fill_between(view.top_k, view.ci95_low, view.ci95_high, color=HIER_SPLIT_COLORS[split], alpha=.16)
        ax.set(title=f'{title}\nGLOBAL | decision_supporting | seed 42', xlabel='Mensagens removidas/acumuladas (k)', ylabel=ylabel)
        ax.grid(alpha=.25); ax.legend(title='Split / IC95%')
    coverage = decision[decision.metric.eq('comprehensiveness_logit_aligned')]
    for split, view in coverage.groupby('split'):
        view = view.sort_values('top_k')
        axes[2].plot(view.top_k, view.n, color=HIER_SPLIT_COLORS[split], marker='o', ms=3, label=split)
    axes[2].set(title='Cobertura da média em cada k\nN varia ao longo da curva', xlabel='Mensagens acumuladas (k)', ylabel='Conversas incluídas (N)')
    axes[2].grid(alpha=.25); axes[2].legend(title='Split')
    plt.tight_layout(); plt.show()
    display(decision[decision.metric.isin(['comprehensiveness_logit_aligned', 'sufficiency_distance_logit'])][['metric', 'top_k', 'n', 'mean', 'std', 'ci95_low', 'ci95_high']])


### 3B. Mesmas métricas pela fração da conversa utilizada

Este gráfico **não é apenas o eixo de 3A convertido para porcentagem**. Cada conversa é reposicionada conforme `k / total de mensagens`, os valores são agrupados em faixas de 5% e cada conversa contribui uma vez por faixa. Assim, a composição e os pesos da média diferem de 3A. O terceiro painel mostra o `N` efetivo de cada faixa.


In [ ]:
if not hierarchical_saturation_fraction_curves.empty:
    decision_fraction = hierarchical_saturation_fraction_curves[
        hierarchical_saturation_fraction_curves.ranking_policy.eq('decision_supporting')
    ]
    fig, axes = plt.subplots(1, 3, figsize=(21, 5))
    for ax, metric, title, ylabel in [
        (axes[0], 'comprehensiveness_logit_aligned', 'Comprehensiveness versus tamanho da conversa', 'Queda alinhada do logit'),
        (axes[1], 'sufficiency_distance_logit', 'Sufficiency versus tamanho da conversa', 'Distância de sufficiency no logit'),
    ]:
        for split, group in decision_fraction[decision_fraction.metric.eq(metric)].groupby('split'):
            group = group.sort_values('removed_fraction_bin')
            ax.plot(100 * group.removed_fraction_bin, group['mean'], marker='o', color=HIER_SPLIT_COLORS[split], label=split)
            ax.fill_between(100 * group.removed_fraction_bin, group.ci95_low, group.ci95_high, color=HIER_SPLIT_COLORS[split], alpha=.14)
        ax.set(title=f'{title}\nGLOBAL | decision_supporting | seed 42', xlabel='% do total de mensagens acumuladas', ylabel=ylabel)
        ax.grid(alpha=.25); ax.legend(title='Split / IC95%')
    coverage = decision_fraction[decision_fraction.metric.eq('comprehensiveness_logit_aligned')]
    for split, group in coverage.groupby('split'):
        group = group.sort_values('removed_fraction_bin')
        axes[2].plot(100 * group.removed_fraction_bin, group.n, marker='o', color=HIER_SPLIT_COLORS[split], label=split)
    axes[2].set(title='Cobertura por faixa proporcional', xlabel='% do total de mensagens acumuladas', ylabel='Conversas incluídas (N)')
    axes[2].grid(alpha=.25); axes[2].legend(title='Split')
    plt.tight_layout(); plt.show()


### 3C. Quantas mensagens são necessárias para atingir k95?

In [ ]:
if not hierarchical_saturation_points.empty:
    reached = hierarchical_saturation_points[hierarchical_saturation_points.status.eq('reached')]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.boxplot(data=reached, x='split', y='k95', palette=HIER_SPLIT_COLORS, showfliers=False, ax=axes[0])
    axes[0].set_title('Mensagens para alcançar 95% do máximo observado'); axes[0].set_ylabel('k95 (mensagens)')
    sns.boxplot(data=reached, x='split', y='k95_fraction_total', palette=HIER_SPLIT_COLORS, showfliers=False, ax=axes[1])
    axes[1].set_title('k95 relativo ao tamanho da conversa'); axes[1].set_ylabel('k95 / total de mensagens')
    axes[1].yaxis.set_major_formatter(lambda value, _: f'{100*value:.0f}%')
    plt.tight_layout(); plt.show()
    display(reached.groupby('split').agg(
        conversas=('sample_id', 'nunique'),
        k95_mediana=('k95', 'median'),
        k95_media=('k95', 'mean'),
        fracao_mediana=('k95_fraction_total', 'median'),
        fracao_media=('k95_fraction_total', 'mean'),
        sufficiency_logit_k95=('sufficiency_distance_logit_at_k95', 'mean'),
    ))


### 3D. Massa aditiva de evidência

Dentro de cada conversa, estas curvas são somas e, portanto, não diminuem. `absolute_influence` soma os módulos dos impactos individuais; `decision_supporting` soma somente impactos alinhados à decisão. Elas **não são comprehensiveness**, pois não reavaliam conjuntamente as interações do Transformer.

A média global ainda pode cair porque `N` varia com `k`: conversas que terminaram saem da média. Logo, uma queda nesta figura é um efeito de composição, não uma perda de massa dentro das conversas. Os dois splits são mostrados separadamente para evitar sobrepor escalas muito diferentes.


In [ ]:
if not hierarchical_saturation_curves.empty:
    mass = hierarchical_saturation_curves[
        hierarchical_saturation_curves.metric.eq('cumulative_evidence_mass_logit')
    ]
    splits = [split for split in ('test_internal', 'validation') if split in set(mass.split)]
    fig, axes = plt.subplots(1, len(splits), figsize=(8 * len(splits), 5), squeeze=False)
    colors = {'decision_supporting': '#16a34a', 'absolute_influence': '#7c3aed'}
    for ax, split in zip(axes[0], splits):
        for policy, group in mass[mass.split.eq(split)].groupby('ranking_policy'):
            group = group.sort_values('top_k')
            ax.plot(group.top_k, group['mean'], marker='o', ms=3, color=colors[policy], label=f'{policy} (N varia por k)')
            ax.fill_between(group.top_k, group.ci95_low, group.ci95_high, color=colors[policy], alpha=.14)
        ax.set_title(f'Massa acumulada — GLOBAL | {split} | seed 42')
        ax.set_xlabel('Quantidade acumulada de mensagens'); ax.set_ylabel('Soma acumulada de impactos no logit')
        ax.grid(alpha=.25); ax.legend(title='Política / IC95%')
    plt.tight_layout(); plt.show()


### 3E. Auditoria qualitativa — três conversas completas de validation

Por padrão são escolhidos deterministicamente: um TP, um TN e um erro de classificação (FN primeiro; FP como fallback), sempre próximos da mediana de `k95 / total de mensagens` de sua categoria. Defina `HIER_VALIDATION_EXAMPLE_IDS` com três IDs para auditar casos específicos.

Cada tabela mantém **todas as mensagens na ordem da conversa**. `rank_decision` ordena somente evidências que sustentam a classe prevista; `selecionada_ate_k95` mostra quais mensagens formam a explicação até o ponto de saturação. `rank_absolute` mantém também impactos contrários à decisão.


In [ ]:
HIER_VALIDATION_EXAMPLE_IDS = None  # exemplo: ['id-1', 'id-2', 'id-3']

def choose_validation_audit_examples(points, requested=None):
    available = points[(points.split.eq('validation')) & points.status.eq('reached')].copy()
    if requested:
        requested = [str(value) for value in requested]
        missing = sorted(set(requested) - set(available.sample_id.astype(str)))
        if missing:
            raise ValueError(f'IDs sem resultado de saturação em validation: {missing}')
        return requested[:3]
    chosen = []
    for category in ('TP', 'TN'):
        group = available[available.prediction_category.eq(category)].copy()
        if not group.empty:
            median = group.k95_fraction_total.median()
            group['distance_to_category_median'] = (group.k95_fraction_total - median).abs()
            chosen.append(str(group.sort_values(['distance_to_category_median', 'sample_id']).iloc[0].sample_id))
    error = available[available.prediction_category.isin(['FN', 'FP'])].copy()
    if not error.empty:
        error['error_priority'] = error.prediction_category.map({'FN': 0, 'FP': 1})
        category = error.sort_values(['error_priority', 'sample_id']).iloc[0].prediction_category
        group = error[error.prediction_category.eq(category)].copy()
        median = group.k95_fraction_total.median()
        group['distance_to_category_median'] = (group.k95_fraction_total - median).abs()
        chosen.append(str(group.sort_values(['distance_to_category_median', 'sample_id']).iloc[0].sample_id))
    if len(chosen) < 3:
        remaining = available[~available.sample_id.astype(str).isin(chosen)].sort_values(['k95_fraction_total', 'sample_id'])
        chosen.extend(remaining.sample_id.astype(str).head(3 - len(chosen)).tolist())
    return chosen[:3]

HIER_VALIDATION_AUDIT_IDS = choose_validation_audit_examples(
    hierarchical_saturation_points,
    HIER_VALIDATION_EXAMPLE_IDS,
) if not hierarchical_saturation_points.empty else []
print('Casos selecionados para auditoria:', HIER_VALIDATION_AUDIT_IDS)


In [ ]:
def validation_message_audit_table(sample_id):
    case = hierarchical_messages[
        (hierarchical_messages.split.eq('validation')) &
        (hierarchical_messages.seed.eq(42)) &
        (hierarchical_messages.sample_id.astype(str).eq(str(sample_id)))
    ].copy()
    point = hierarchical_saturation_points[
        (hierarchical_saturation_points.split.eq('validation')) &
        (hierarchical_saturation_points.sample_id.astype(str).eq(str(sample_id)))
    ].iloc[0]
    case['rank_absolute'] = case.abs_delta_logit.rank(method='first', ascending=False).astype(int)
    supporting = case.class_support_delta.gt(1e-4)
    case['rank_decision'] = pd.Series(pd.NA, index=case.index, dtype='Int64')
    case.loc[supporting, 'rank_decision'] = case.loc[supporting, 'class_support_delta'].rank(method='first', ascending=False).astype(int)
    case['selecionada_ate_k95'] = case.rank_decision.notna() & case.rank_decision.le(int(point.k95))
    case['papel_na_decisao'] = np.where(
        case.class_support_delta.gt(1e-4), 'sustenta decisão',
        np.where(case.class_support_delta.lt(-1e-4), 'contraria decisão', 'neutra'),
    )
    return case.sort_values('model_index')[['model_index', 'speaker', 'rank_decision', 'rank_absolute', 'selecionada_ate_k95', 'papel_na_decisao', 'class_support_delta', 'delta_logit', 'text']], point

def style_validation_audit(frame):
    def row_style(row):
        if bool(row.selecionada_ate_k95):
            return ['background-color: #fee2e2'] * len(row)
        if row.papel_na_decisao == 'contraria decisão':
            return ['background-color: #dbeafe'] * len(row)
        return [''] * len(row)
    return (
        frame.style.apply(row_style, axis=1)
        .format({'class_support_delta': '{:.4f}', 'delta_logit': '{:.4f}'})
        .set_properties(subset=['text'], **{'white-space': 'pre-wrap', 'text-align': 'left', 'max-width': '720px'})
    )

for audit_number, sample_id in enumerate(HIER_VALIDATION_AUDIT_IDS, start=1):
    table, point = validation_message_audit_table(sample_id)
    prediction = 'Scam' if int(point.predicted_label) else 'Ham'
    metadata = pd.DataFrame([{
        'caso': audit_number,
        'sample_id': sample_id,
        'categoria': point.prediction_category,
        'classe_prevista': prediction,
        'total_mensagens': int(point.total_messages),
        'k95': int(point.k95),
        'k95_percentual_conversa': 100 * float(point.k95_fraction_total),
        'comprehensiveness_logit_k95': float(point.comprehensiveness_logit_at_k95),
        'sufficiency_distance_logit_k95': float(point.sufficiency_distance_logit_at_k95),
    }])
    print(f'VALIDATION — CASO {audit_number} — {point.prediction_category} — {sample_id}')
    display(metadata.style.format({'k95_percentual_conversa': '{:.1f}%', 'comprehensiveness_logit_k95': '{:.4f}', 'sufficiency_distance_logit_k95': '{:.4f}'}))
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
        display(style_validation_audit(table))

    ranked = table[table.selecionada_ate_k95].sort_values('rank_decision', ascending=False)
    case_curve = hierarchical_validation_saturation[
        (hierarchical_validation_saturation.sample_id.astype(str).eq(str(sample_id))) &
        hierarchical_validation_saturation.ranking_policy.eq('decision_supporting')
    ].sort_values('top_k')
    fig, axes = plt.subplots(1, 2, figsize=(17, max(4.5, .38 * max(len(ranked), 8))))
    axes[0].barh(
        ranked.apply(lambda row: f"#{int(row.rank_decision)} · msg {int(row.model_index)} · {str(row.text)[:90]}", axis=1),
        ranked.class_support_delta,
        color='#dc2626' if prediction == 'Scam' else '#2563eb',
    )
    axes[0].set_title(f'Mensagens selecionadas até k95={int(point.k95)}'); axes[0].set_xlabel('Suporte individual alinhado no logit')
    axes[1].plot(case_curve.top_k, case_curve.comprehensiveness_logit_aligned, marker='o', label='Comprehensiveness logit')
    axes[1].plot(case_curve.top_k, case_curve.sufficiency_distance_logit, marker='s', label='Sufficiency distance logit')
    axes[1].axvline(int(point.k95), color='black', ls='--', label='k95')
    axes[1].set(title=f'Curvas desta conversa ({int(point.total_messages)} mensagens)', xlabel='Quantidade acumulada de mensagens', ylabel='Magnitude no logit')
    axes[1].grid(alpha=.25); axes[1].legend()
    plt.tight_layout(); plt.show()


### 4–5. Distribuições globais por split e Top-k

In [ ]:
if not hierarchical_metrics.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    sns.boxplot(data=hierarchical_metrics, x='top_k', y='comprehensiveness', hue='split', palette=HIER_SPLIT_COLORS, showfliers=False, ax=axes[0])
    axes[0].axhline(0, color='black', lw=1); axes[0].set_title('Comprehensiveness — GLOBAL | decision_supporting | seed 42')
    sns.boxplot(data=hierarchical_metrics, x='top_k', y='sufficiency_distance', hue='split', palette=HIER_SPLIT_COLORS, showfliers=False, ax=axes[1])
    axes[1].set_title('Sufficiency distance — GLOBAL | decision_supporting | seed 42 (sempre ≥ 0)')
    for ax in axes: ax.set_xlabel('Top-k'); ax.legend(title='Split')
    plt.tight_layout(); plt.show()


### 6. Variação por TP, TN, FP e FN

In [ ]:
if not hierarchical_metrics.empty:
    fig, axes = plt.subplots(2, 2, figsize=(17, 10), sharex=True)
    for column, split in enumerate(('test_internal', 'validation')):
        view = hierarchical_metrics[hierarchical_metrics.split.eq(split)]
        sns.boxplot(data=view, x='top_k', y='comprehensiveness', hue='prediction_category', palette=HIER_CATEGORY_COLORS, showfliers=False, ax=axes[0, column])
        axes[0, column].axhline(0, color='black', lw=1); axes[0, column].set_title(f'{split} — comprehensiveness')
        sns.boxplot(data=view, x='top_k', y='sufficiency_distance', hue='prediction_category', palette=HIER_CATEGORY_COLORS, showfliers=False, ax=axes[1, column])
        axes[1, column].set_title(f'{split} — sufficiency distance')
    fig.suptitle('GLOBAL | decision_supporting | seed 42 | cores = TP/TN/FP/FN')
    plt.tight_layout(); plt.show()


### 7–8. Trechos selecionados versus controles aleatórios

In [ ]:
if not hierarchical_controls.empty:
    unique_controls = hierarchical_controls.drop_duplicates(['split', 'sample_id', 'seed', 'model_index', 'start_char', 'end_char'])
    impact = pd.concat([
        unique_controls[['split', 'prediction_category', 'selected_abs_delta_logit']].rename(columns={'selected_abs_delta_logit': 'impact'}).assign(tipo='Trecho selecionado'),
        hierarchical_controls[['split', 'prediction_category', 'control_abs_delta_logit']].rename(columns={'control_abs_delta_logit': 'impact'}).assign(tipo='Controle aleatório'),
    ], ignore_index=True).dropna()
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.boxplot(data=impact, x='split', y='impact', hue='tipo', showfliers=False, ax=axes[0])
    axes[0].set_title(f'Impacto |Δ logit| — GLOBAL | seed 42 | N={len(unique_controls):,} trechos')
    sns.boxplot(data=unique_controls, x='split', y='specificity_ratio', hue='prediction_category', palette=HIER_CATEGORY_COLORS, showfliers=False, ax=axes[1])
    axes[1].axhline(1, color='black', ls='--', label='Mesmo impacto dos controles')
    axes[1].set_title('Razão de especificidade — GLOBAL | decision_supporting | seed 42')
    axes[1].legend(title='Categoria / referência')
    plt.tight_layout(); plt.show()
    global_control_ci = hierarchical_control_summary[hierarchical_control_summary.aggregation.eq('population')]
    display(global_control_ci[['split', 'metric', 'n', 'mean', 'std', 'ci95_low', 'ci95_high']])


### 9–10. Estabilidade global entre seeds 42, 52 e 62

In [ ]:
if not hierarchical_stability.empty:
    summary_stability = hierarchical_stability[hierarchical_stability.item_type.eq('summary')].copy()
    metrics = ['message_top_k_jaccard', 'message_direction_agreement', 'message_rank_spearman', 'message_rank_kendall', 'span_top_k_jaccard', 'span_direction_agreement', 'span_rank_kendall']
    long = summary_stability.melt(id_vars=['split', 'sample_id', 'compared_seed', 'prediction_agreement'], value_vars=[m for m in metrics if m in summary_stability], var_name='métrica', value_name='valor').dropna()
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    sns.boxplot(data=long, x='métrica', y='valor', hue='split', palette=HIER_SPLIT_COLORS, showfliers=False, ax=axes[0])
    axes[0].tick_params(axis='x', rotation=20); axes[0].set_title(f'Estabilidade de rankings — GLOBAL | seeds 42×52/62 | N={summary_stability.sample_id.nunique():,} conversas')
    agreement = summary_stability.groupby(['split', 'compared_seed'], as_index=False).prediction_agreement.mean()
    sns.barplot(data=agreement, x='compared_seed', y='prediction_agreement', hue='split', palette=HIER_SPLIT_COLORS, ax=axes[1])
    axes[1].set_ylim(0, 1); axes[1].set_title('Concordância da classe prevista (discordâncias excluídas das métricas de justificativa)')
    plt.tight_layout(); plt.show()
    item_stability = hierarchical_stability[hierarchical_stability.item_type.isin(['message', 'span'])].dropna(subset=['impact_coefficient_variation'])
    if not item_stability.empty:
        sns.boxplot(data=item_stability, x='item_type', y='impact_coefficient_variation', hue='split', palette=HIER_SPLIT_COLORS, showfliers=False)
        plt.title('Coeficiente de variação do impacto — GLOBAL | seeds 42/52/62')
        plt.xlabel('Elemento'); plt.ylabel('desvio-padrão / média |Δ logit|'); plt.legend(title='Split'); plt.show()


### 11. Explorador por conversa

Defina `HIER_STUDY_SAMPLE_ID` e `HIER_STUDY_SPLIT`. A esquerda ficam as seis mensagens por influência absoluta; a direita, apenas justificativas alinhadas à classe prevista. Os painéis inferiores são **desta conversa**, não médias globais.


In [ ]:
HIER_STUDY_SPLIT = 'test_internal'
HIER_STUDY_SAMPLE_ID = None
HIER_STUDY_SEED = 42

available = (
    hierarchical_messages[(hierarchical_messages.split.eq(HIER_STUDY_SPLIT)) & (hierarchical_messages.seed.eq(HIER_STUDY_SEED))]
    if not hierarchical_messages.empty else pd.DataFrame()
)
if not available.empty:
    if HIER_STUDY_SAMPLE_ID is None:
        HIER_STUDY_SAMPLE_ID = str(available.sample_id.iloc[0])
    case_messages = available[available.sample_id.astype(str).eq(str(HIER_STUDY_SAMPLE_ID))]
    case_spans = hierarchical_spans[(hierarchical_spans.split.eq(HIER_STUDY_SPLIT)) & (hierarchical_spans.seed.eq(HIER_STUDY_SEED)) & (hierarchical_spans.sample_id.astype(str).eq(str(HIER_STUDY_SAMPLE_ID)))]
    case_fidelity = hierarchical_metrics[(hierarchical_metrics.split.eq(HIER_STUDY_SPLIT)) & (hierarchical_metrics.sample_id.astype(str).eq(str(HIER_STUDY_SAMPLE_ID)))]
    absolute = case_messages[case_messages.selected_absolute_influence].nlargest(6, 'abs_delta_logit').sort_values('delta_logit')
    supporting = case_messages[case_messages.selected_decision_supporting].sort_values('class_support_delta').tail(6)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    axes[0].barh(absolute.text.str.slice(0, 75), absolute.delta_logit, color=np.where(absolute.delta_logit.ge(0), '#dc2626', '#2563eb'))
    axes[0].set_title(f'CONVERSA {HIER_STUDY_SAMPLE_ID}: Top-6 absolute_influence\nvermelho=pró-Scam; azul=pró-Ham')
    axes[1].barh(supporting.text.str.slice(0, 75), supporting.class_support_delta, color='#16a34a')
    supported = supporting.supported_class.iloc[0] if len(supporting) else 'sem evidência válida'
    axes[1].set_title(f'Justificativas decision_supporting para {supported}\nseed 42 | N={len(supporting)}')
    plt.tight_layout(); plt.show()

    selected_indices = set(supporting.model_index.astype(int))
    span_view = case_spans[case_spans.model_index.isin(selected_indices) & case_spans.class_support_delta.gt(1e-4)]
    span_view = span_view.sort_values('class_support_delta').groupby('model_index', group_keys=False).tail(3)
    if not span_view.empty:
        plt.figure(figsize=(12, max(4, len(span_view) * .35)))
        plt.barh(span_view.span_text.str.slice(0, 90), span_view.class_support_delta, color='#dc2626' if supported == 'scam' else '#2563eb')
        plt.title(f'N-grams alinhados à decisão — CONVERSA {HIER_STUDY_SAMPLE_ID} | {HIER_STUDY_SPLIT} | seed 42 | N={len(span_view)}')
        plt.xlabel('class_support_delta'); plt.tight_layout(); plt.show()

    if not case_fidelity.empty:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        axes[0].plot(case_fidelity.top_k, case_fidelity.comprehensiveness, marker='o', label='Comprehensiveness')
        axes[0].axhline(0, color='black', lw=1); axes[0].legend(); axes[0].set_title('CONVERSA: remoção cumulativa')
        axes[1].plot(case_fidelity.top_k, case_fidelity.sufficiency_distance, marker='s', color='#f97316', label='Sufficiency distance')
        axes[1].legend(); axes[1].set_title('CONVERSA: somente evidências')
        for ax in axes: ax.set_xlabel('Top-k'); ax.grid(alpha=.25)
        plt.tight_layout(); plt.show()

    case_controls = hierarchical_controls[(hierarchical_controls.split.eq(HIER_STUDY_SPLIT)) & (hierarchical_controls.sample_id.astype(str).eq(str(HIER_STUDY_SAMPLE_ID)))]
    case_stability = hierarchical_stability[(hierarchical_stability.split.eq(HIER_STUDY_SPLIT)) & (hierarchical_stability.sample_id.astype(str).eq(str(HIER_STUDY_SAMPLE_ID))) & hierarchical_stability.item_type.eq('summary')]
    display(case_controls.drop_duplicates(['model_index', 'start_char', 'end_char'])[['span_text', 'selected_abs_delta_logit', 'control_mean', 'specificity_ratio']])
    display(case_stability)


## Checklist de interpretação

1. Leia `N`/cobertura junto com toda curva global; médias em `k` diferentes podem conter populações diferentes.
2. Use comprehensiveness e sufficiency em conjunto: uma explicação boa tende a remover suporte quando retirada e a preservar a decisão quando isolada.
3. Use os boxplots para verificar heterogeneidade; a média e o IC95% não mostram casos individuais extremos.
4. Controles aleatórios avaliam especificidade, não veracidade factual do trecho.
5. Estabilidade entre seeds mede robustez do modelo, não causalidade no mundo real.
6. No explorador, `absolute_influence` é diagnóstico; `decision_supporting` é a política usada para justificar a classe prevista.
